In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
# Parameters for the symmetric Gaussians
mu = 2.2   # distance from y=0
sigma = 5.0
x = np.linspace(-15, 15, 500)

# Two symmetric Gaussian PDFs
pdf_left = norm.pdf(x, -mu, sigma)
pdf_right = norm.pdf(x, mu, sigma)

# Log ratio of PDFs
log_ratio = np.log10(pdf_right / pdf_left)

# Define target log ratio values
# target_log_ratios = np.array([-0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
# target_log_ratios = np.array([-0.9, -0.7, -0.5, -0.3, -0.1, 0.1, 0.3, 0.5, 0.7, 0.9])
target_log_ratios = np.array([-0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
# target_log_ratios = np.array([-0.3, 0.3])

# Find corresponding x values by interpolation
x_targets = np.interp(target_log_ratios, log_ratio, x)

# Plot with vertical lines at target x values
fig, ax = plt.subplots(3, 1, figsize=(2.5, 3.8), gridspec_kw={'height_ratios': [2, 1, 2]})

pB_raw = norm.pdf(x_targets, mu, sigma)
pA_raw = norm.pdf(x_targets, -mu, sigma)

# Normalize
pB = pB_raw / pB_raw.sum()
pA = pA_raw / pA_raw.sum()

# --- New: set error rates ---
alpha = 0.05   # P_A(x > t_alpha) = α
beta  = 0.05   # P_B(x < t_beta)  = β

# --- Compute thresholds from quantiles ---
t_alpha = norm.ppf(1 - alpha, loc=-mu, scale=sigma)  # right-tail under A
t_beta  = norm.ppf(beta,      loc=+mu, scale=sigma)  # left-tail  under B

# Top plot: PDFs
ax[0].plot(x, pdf_left, label='Target A', color='blue')
ax[0].plot(x, pdf_right, label='Target B', color='red')

# # --- New: shade α and β error areas ---
# # α: under A ~ N(-mu, sigma), right tail x >= t_alpha (blue shade)
# ax[0].fill_between(x, 0, pdf_left, where=(x >= t_alpha), alpha=0.25, color='blue', label=r'$\alpha$ area')

# # β: under B ~ N(+mu, sigma), left tail x <= t_beta (red shade)
# ax[0].fill_between(x, 0, pdf_right, where=(x <= t_beta),  alpha=0.25, color='red',  label=r'$\beta$ area')

# draw decision thresholds as vertical lines (optional but helpful)
# ax[0].axvline(t_alpha, color='blue', linestyle=':', linewidth=1.5)
# ax[0].axvline(t_beta,  color='red',  linestyle=':', linewidth=1.5)

# keep your target x vertical lines
for xt in x_targets:
    ax[0].axvline(xt, color='gray', linestyle='--', alpha=0.7)

ax[0].set_xlim(-12, 12)
ax[0].set_xticks(range(-12, 13, 4))
ax[0].set_ylabel('PDF')
ax[0].legend()
ax[0].set_title('Task Distribution')

# middle plot: Log ratio
ax[1].plot(x, log_ratio, color='green')
for xt in x_targets:
    ax[1].axvline(xt, color='gray', linestyle='--', alpha=0.7)
ax[1].axhline(0, color='black', linestyle='--')
ax[1].set_xlim(-12, 12)
ax[1].set_xticks(range(-12, 13, 4))
# ax[1].set_xlabel('x')
ax[1].set_ylabel('logLR')

# bottom plot: Probability
markerline1, stemlines1, baseline1 = ax[2].stem(x_targets, pA, linefmt='b-', markerfmt='bo', basefmt='k-', label='Target A')
markerline2, stemlines2, baseline2 = ax[2].stem(x_targets, pB, linefmt='r-', markerfmt='ro', basefmt='k-', label='Target B')
markerline1.set_markersize(4)
markerline2.set_markersize(4)
ax[2].legend()
ax[2].set_xlim(-12, 12)
ax[2].set_xticks(range(-12, 13, 4))
# ax[2].set_ylim(0, 0.25)
ax[2].set_xlabel('x')
ax[2].set_ylabel('PMF')

# hide the xticks
ax[2].set_xticks([])
# hide the xlabel
ax[2].set_xlabel('')

leg2 = ax[2].get_legend()
leg2.get_frame().set_linewidth(0.5)
for lh in leg2.get_lines():
    lh.set_linewidth(1.0)
    lh.set_markersize(3.0)

plt.tight_layout()

# save the figure to svg
plt.savefig('distribution.svg', dpi=300)

plt.show()

# ============================================================
# Finite-deadline optimal stopping by backward induction
# Deadline: t = 11
# Terminal value: V_11(b) = 0 everywhere
# Costs: c = .01, .02, .03, .04, .05
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Belief grid
N_b = 1001
b_grid = np.linspace(0, 1, N_b)

# Time grid
T_deadline = 11
t_grid = np.arange(T_deadline + 1)  # 0,...,11

# Terminal rewards
R_correct = 1.0
R_wrong = 0.0

# Stopping value for t < 11
G = np.maximum(
    b_grid * R_correct + (1 - b_grid) * R_wrong,
    (1 - b_grid) * R_correct + b_grid * R_wrong
)

# At the deadline, value is fixed at zero everywhere
G_deadline = np.zeros_like(b_grid)

c_values = [0.01, 0.02, 0.03, 0.04, 0.05]

all_lower_boundaries = {}
all_upper_boundaries = {}
all_V = {}

for c in c_values:

    # V[t, j] = value at time t and belief b_grid[j]
    V = np.zeros((T_deadline + 1, N_b))

    # Deadline condition
    V[T_deadline, :] = G_deadline

    lower_boundary = np.full(T_deadline + 1, np.nan)
    upper_boundary = np.full(T_deadline + 1, np.nan)

    # At t = 11, no stopping boundary is meaningful because value is fixed at 0
    lower_boundary[T_deadline] = np.nan
    upper_boundary[T_deadline] = np.nan

    # Backward induction: t = 10, 9, ..., 0
    for t in range(T_deadline - 1, -1, -1):

        continue_value = np.zeros_like(b_grid)

        for j, b in enumerate(b_grid):

            expected_future_value = 0.0

            # If t >= 10, assume no new information arrives.
            # The agent can only move to t+1 and pay cost c.
            if t >= 10:
                expected_future_value = np.interp(b, b_grid, V[t + 1, :])

            else:
                # Bayesian predictive update with new observation
                for i in range(len(pA)):

                    q_i = b * pB[i] + (1 - b) * pA[i]

                    if q_i > 0:
                        b_next = (b * pB[i]) / q_i

                        expected_future_value += q_i * np.interp(
                            b_next,
                            b_grid,
                            V[t + 1, :]
                        )

            continue_value[j] = -c + expected_future_value

        # Stop value is normal G(b) for t < 11
        stop_value = G

        # Bellman update
        V[t, :] = np.maximum(stop_value, continue_value)

        # Continue region
        continue_region = continue_value > stop_value

        if np.any(continue_region):
            lower_boundary[t] = b_grid[continue_region][0]
            upper_boundary[t] = b_grid[continue_region][-1]
        else:
            # No continuation region
            lower_boundary[t] = 0.5
            upper_boundary[t] = 0.5

    all_lower_boundaries[c] = lower_boundary
    all_upper_boundaries[c] = upper_boundary
    all_V[c] = V

    print(f"\nc = {c:.2f}")
    for t in range(T_deadline + 1):
        print(
            f"t={t:2d}: "
            f"lower={lower_boundary[t]}, "
            f"upper={upper_boundary[t]}"
        )


# ============================================================
# Plot optimal stopping boundaries
# ============================================================

plt.figure(figsize=(8, 5))

for c in c_values:

    plt.plot(
        t_grid,
        all_lower_boundaries[c],
        marker="o",
        label=f"Lower boundary, c={c:.2f}"
    )

    plt.plot(
        t_grid,
        all_upper_boundaries[c],
        marker="s",
        linestyle="--",
        label=f"Upper boundary, c={c:.2f}"
    )

plt.xlabel("Time step t")
plt.ylabel(r"Belief $b=P(H_1 \mid evidence)$")
plt.title("Finite-deadline optimal stopping boundaries by backward induction")
plt.ylim(0, 1)
plt.xticks(t_grid)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

max_timestep = 15

fig, axes = plt.subplots(1, max_timestep, figsize=(7, 3), sharey=True)
if max_timestep == 1:
    axes = [axes]

df["evidences"] = df.apply(
    lambda row: [target_log_ratios[s] for s in row["stimuli_so_far"][:row["decision_time"]]],
    axis=1
)
df["evidence_sum"] = df["evidences"].apply(sum)

colors = {0:'blue', 1:'red', 2:'green'}
labels = {0:'Choice A (0)', 1:'Choice B (1)', 2:'Sampling (2)'}

for t in range(max_timestep):
    ax = axes[t]

    if t >= 10:
        timestep_data = df[df['decision_time'] == t+1]
    else:
        timestep_data = df[df['time_step'] == t+1]

    for action in [0,1,2]:
        action_data = timestep_data[timestep_data['action'] == action]
        if action_data.empty:
            continue

        ev = action_data['evidence_sum'].round(10).to_numpy()
        uniq_vals, counts = np.unique(ev, return_counts=True)

        ax.fill_betweenx(uniq_vals, 0, counts,
                         color=colors[action], alpha=0.3,
                         label=labels[action] if t==0 else None)

        if action in [0,1]:
            mean_ev = action_data['evidence_sum'].mean()
            ax.axhline(mean_ev, color=colors[action], linewidth=2)

    ax.set_xticks([0, max(counts)]) 
    ax.set_xlabel(str(t+1))
    ax.set_xlim(0, None)
    ax.set_ylim(-3.2, 3.2)

fig.text(0.54, 0.03, 'Timestep', ha='center', fontsize=8)
fig.text(0.1, 0.5, 'Cumulative Evidence', va='center', rotation='vertical', fontsize=8)

for ax in axes:
    ax.set_xticks([])
    ax.label_outer()

axes[0].legend(loc='best')

plt.tight_layout(rect=[0.05, 0.05, 1, 1])
plt.show()